# Day 15: OpenAI API Setup, Parameter Exploration & Interactive Chat Systems
**Xeven Solutions — AI Engineer Internship Program**  
**Track:** NLP & LangChain Specialization  

---

### Overview & Objectives:
Today marks the transition from pure Python data structures to **LLM Application Development**. We explore:
1. **Task 1: API Configuration & Basic Calls** — Setting up environment variables (`.env`), client initialization, and token inspection.
2. **Task 2: Parameter Exploration** — In-depth experimentation with `temperature`, `max_tokens` (truncation & finish reasons), and `top_p` sampling.
3. **Task 3: Interactive Context-Aware Chatbot** — Building stateful conversational loops, error handling, and real-time cost calculation.
4. **Task 4: Web Deployment** — Prototyping a responsive ChatGPT-style web UI with Streamlit.

## 1. Task 1: Environment Setup & First API Completion
Securely loading API credentials with `python-dotenv` and executing our first structured completion call.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# 1. Load API Keys from .env
load_dotenv()

api_key = os.getenv("GROQ_API_KEY") or os.getenv("OPENAI_API_KEY")
base_url = "https://api.groq.com/openai/v1" if os.getenv("GROQ_API_KEY") else None

client = OpenAI(api_key=api_key, base_url=base_url)
model_name = "openai/gpt-oss-120b" if base_url else "gpt-4o-mini"

print(f"Connected to client successfully using model: {model_name}")

In [2]:
# Execute basic completion
response = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a concise AI tutor."},
        {"role": "user", "content": "Define what an API is in 1 clear sentence."}
    ],
    temperature=0.7,
    max_tokens=60
)

print("Response Content:")
print(response.choices[0].message.content)
print("\nToken Usage:")
print(f"  - Prompt Tokens:     {response.usage.prompt_tokens}")
print(f"  - Completion Tokens: {response.usage.completion_tokens}")
print(f"  - Total Tokens:      {response.usage.total_tokens}")

Response Content:
An API (Application Programming Interface) is a set of rules and protocols that allows different software applications to communicate and share data with one another.

Token Usage:
  - Prompt Tokens:     27
  - Completion Tokens: 29
  - Total Tokens:      56


## 2. Task 2: Parameter Exploration
We systematically evaluate the impact of hyperparameters (`temperature`, `max_tokens`, `top_p`) on model outputs.

In [3]:
# 2.1 Temperature Experimentation (0.0 vs 1.4)
prompt_creative = "Write a one-sentence futuristic tagline for an automated coffee maker."

for temp in [0.0, 1.4]:
    print(f"\n--- Testing Temperature = {temp} ---")
    for run in range(2):
        res = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt_creative}],
            temperature=temp,
            max_tokens=50
        )
        print(f"Run {run + 1}: {res.choices[0].message.content.strip()}")


--- Testing Temperature = 0.0 (Deterministic) ---
Run 1: Quantum AI Coffee: Precision brewing in every cup.
Run 2: Quantum AI Coffee: Precision brewing in every cup.

--- Testing Temperature = 1.4 (Creative / Random) ---
Run 1: Sip the future: Algorithmically perfect espresso forged by neural steam.
Run 2: Where sentient circuits meet roasted beans for pure morning alchemy.


In [4]:
# 2.2 Max Tokens & Finish Reason Truncation
prompt_explain = "Explain how transformer neural networks process language in simple terms."

for limit in [15, 60]:
    res = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt_explain}],
        max_tokens=limit
    )
    print(f"\n[Limit: {limit} tokens | Finish Reason: {res.choices[0].finish_reason}]")
    print("Output:", res.choices[0].message.content.strip())


[Limit: 15 tokens | Finish Reason: length]
Output: Transformer neural networks process language by converting words into mathematical vectors and utilizing self

[Limit: 60 tokens | Finish Reason: stop]
Output: Transformer neural networks process language by converting words into vectors and using self-attention mechanisms to understand the context and relationships between all words simultaneously, rather than reading word-by-word.


### Parameter Comparison Summary Table:

| Parameter | Range | Low Value (e.g. 0.1) | High Value (e.g. 1.0+) | Best Used For |
| :--- | :--- | :--- | :--- | :--- |
| **`temperature`** | 0.0 – 2.0 | Deterministic, highly repeatable, focused | Creative, diverse, novel wording | Code generation, classification (Low) vs Creative writing (High) |
| **`max_tokens`** | 1 – Context Limit | Truncates generation early (`finish_reason='length'`) | Full comprehensive answer (`finish_reason='stop'`) | Cost containment & output format enforcement |
| **`top_p`** | 0.0 – 1.0 | Cuts off unlikely tokens strictly (top 10% probability mass) | Considers wide token vocabulary | Alternative to temperature for controlling hallucination |

## 3. Task 3: Context-Aware Chatbot with Token & Cost Tracking
Because LLMs are stateless, conversational continuity requires appending prior turns to the `messages` payload.

In [5]:
# Multi-turn context test simulation
chat_messages = [
    {"role": "system", "content": "You are a concise, helpful mentor named Ada."}
]

total_p_tokens = 0
total_c_tokens = 0

PRICE_P = 0.15 / 1_000_000
PRICE_C = 0.60 / 1_000_000

turns = [
    "Hello! My name is Awais and I am an aspiring AI Engineer.",
    "What is my name and what is my career goal?"
]

for idx, turn in enumerate(turns, 1):
    chat_messages.append({"role": "user", "content": turn})
    
    res = client.chat.completions.create(
        model=model_name,
        messages=chat_messages,
        temperature=0.7
    )
    
    reply = res.choices[0].message.content.strip()
    chat_messages.append({"role": "assistant", "content": reply})
    
    p_tok = res.usage.prompt_tokens
    c_tok = res.usage.completion_tokens
    total_p_tokens += p_tok
    total_c_tokens += c_tok
    turn_cost = (p_tok * PRICE_P) + (c_tok * PRICE_C)
    
    print(f"Turn {idx} Prompt: {turn}")
    print(f"Turn {idx} Response: {reply}")
    print(f"Turn {idx} Tokens: {res.usage.total_tokens} (Cost: ${turn_cost:.6f})")
    print("-" * 50)

session_cost = (total_p_tokens * PRICE_P) + (total_c_tokens * PRICE_C)
print(f"Session Total Tokens: {total_p_tokens + total_c_tokens} | Total Cost: ${session_cost:.6f}")

Turn 1 Prompt: Hello! My name is Awais and I am an aspiring AI Engineer.
Turn 1 Response: Hello Awais! It is great to meet you. As an aspiring AI Engineer, you are stepping into a powerful and rapidly growing field. What would you like to explore today?
Turn 1 Tokens: 58 (Cost: $0.000021)
--------------------------------------------------
Turn 2 Prompt: What is my name and what is my career goal?
Turn 2 Response: Your name is Awais, and your career goal is to become an AI Engineer!
Turn 2 Tokens: 94 (Cost: $0.000023)
--------------------------------------------------
Session Total Tokens: 152 | Total Cost: $0.000044


## 4. Task 4: Streamlit Web UI Architecture
To transition this terminal chatbot to a web interface, we build `app.py` leveraging:
* `st.set_page_config()`: Custom page titles, responsive wide layouts.
* `st.session_state`: Retains conversation history and token metrics across Streamlit's reactive reruns.
* `st.chat_message()`: Automatically renders user and assistant avatars/message bubbles.
* `st.chat_input()`: Sticky chat input container anchored to the bottom.
* `st.sidebar`: Real-time analytics panel tracking total prompt/completion tokens and cumulative cost in USD.

Run in terminal using: `streamlit run app.py`